# Check MimicGen source demos before training

Grade [`source/stack.hdf5`](https://huggingface.co/datasets/amandlek/mimicgen_datasets/blob/89835eddc2abbe2966e43024504b1ff62f914a75/source/stack.hdf5) from `amandlek/mimicgen_datasets`, 10 human source demos for the stack task, about 11.8 MB.
CPU only, no API key, nothing is published. Choose **Runtime → Run all**.

## Install

Kalanos with HDF5 support, and plotly, at pinned versions.

In [ ]:
# @title Install Kalanos
%pip install -q --no-warn-conflicts \
    "kalanos[numeric,hdf5,hf]==0.10.2" plotly==7.1.0

## Grade it

Kalanos reads the file from the Hugging Face Hub at a pinned revision, with the `numeric-core-v1` profile at the standard tier.
The CLI prints a report card and saves the full report as JSON: 9 of 10 demos go to review.

In [ ]:
# @title Choose the file
# A fixed revision keeps the file, and so every result below, the same on each run.
SOURCE = "hf://datasets/amandlek/mimicgen_datasets@89835eddc2abbe2966e43024504b1ff62f914a75/source/stack.hdf5"

In [ ]:
# @title Grade it with the Kalanos CLI
# `--verbosity error` hides warnings about file formats this dataset doesn't use.
# Piping through `cat` stops the CLI from animating a progress spinner in the cell,
# and `--color` keeps the card's colours through the pipe.
!kalanos --verbosity error grade {SOURCE} --profile numeric-core-v1 --tier standard --report audit.json --color 2>&1 | cat

## The demos

One row per demo, with its length in action steps and Kalanos's decision.

In [ ]:
# @title Load the report into a table
import polars as pl

import kalanos


loaded = kalanos.load_report("audit.json")
# `load_report` also reads reports from older schemas, which have no episodes.
# Assigning after the check gives `report` that narrower type in later cells too.
assert isinstance(loaded, kalanos.Report)
report = loaded

rows = [
    {
        # Kalanos names each demo `stack.hdf5::data_demo_N`.
        "demo": int(e.id.rsplit("_", 1)[1]),
        "steps": e.sample_count,
        "action_dims": len(
            next(s for s in e.streams if s.source_field == "actions").channels
        ),
        "decision": e.eligibility.status.value,
    }
    for e in report.episodes
    # Every episode in a saved report has a decision; this only tells the type checker.
    if e.eligibility is not None
]

pl.DataFrame(rows).sort("demo")

## Where the demos hold still

In 9 demos, a few simulator channels (`object_*`, `states_*`) hold the same value for more than 90% of the demo, which alone sends them to review ([how decisions work](https://github.com/KalanosAI/kalanos/blob/v0.10.2/docs/DECISIONS.md)).
The file doesn't say what these channels measure, so Kalanos leaves them unmapped (`--map` types them). Kalanos calls this a heuristic: a reason to look, not proof of a fault.

In [ ]:
# @title Chart where each demo holds still
import plotly.graph_objects as go


# Pass and review differ in hue but barely in lightness, so the legend and the callout name each.
PASS_COLOR, REVIEW_COLOR = "#23735e", "#87511b"
PAPER, SURFACE, RULE, INK, MUTED = "#f4f1ea", "#ffffff", "#d2cec5", "#171717", "#65625c"
MONO = "'IBM Plex Mono', monospace"

policy = kalanos.load_policy(None)
review_above = policy.metrics["integrity.flatline_pct"].thresholds["default"].bad

# The channels Kalanos sends to review for holding still, found in the report
# as finding → stream and channel. Sorted, so a tie below always picks the same one.
flagged = sorted(
    {
        (f.source_field, f.channel)
        for f in report.findings
        if f.metric_id == "integrity.flatline_pct" and f.consequence.value == "review"
    }
)

# Per demo, the flagged channel with the longest still stretch,
# found as episode → stream → channel → metric.
demos = []
for e in report.episodes:
    longest = None
    for field, name in flagged:
        stream = next(s for s in e.streams if s.source_field == field)
        channel = next(c for c in stream.channels if c.channel.name == name)
        metric = channel.metrics.get("flatline_pct")
        evidence = (metric.evidence or {}) if metric else {}
        # A channel with no measurement is skipped, never plotted as zero.
        if (
            metric is None
            or metric.value is None
            or "longest_run_start" not in evidence
        ):
            continue
        start = evidence["longest_run_start"]
        end = evidence["longest_run_end_exclusive"]
        if longest is None or end - start > longest["end"] - longest["start"]:
            longest = {
                "channel": name,
                "start": start,
                "end": end,
                "share": metric.value,
            }
    if longest is not None:
        decision = e.eligibility.status.value if e.eligibility else "unknown"
        demo = int(e.id.rsplit("_", 1)[1])
        demos.append(
            {"demo": demo, "steps": e.sample_count, "decision": decision, **longest}
        )
demos.sort(key=lambda d: d["demo"])

# The chart's story is that holding still from the first step sends a demo to review.
# Stop here if the report says otherwise, rather than draw a misleading chart.
charted = {d["demo"] for d in demos}
missing = [e.id for e in report.episodes if int(e.id.rsplit("_", 1)[1]) not in charted]
if missing:
    raise ValueError(f"No held-still measurement to chart for {missing}.")
if sum(d["decision"] == "pass" for d in demos) != 1:
    raise ValueError("Expected exactly one passing demo.")
for d in demos:
    if d["decision"] not in ("pass", "review"):
        raise ValueError(f"Demo {d['demo']} has decision {d['decision']}.")
    if d["decision"] == "review" and (d["share"] <= review_above or d["start"] != 0):
        raise ValueError(
            f"Demo {d['demo']} is in review without holding still from step 0."
        )
    if d["decision"] == "pass" and d["share"] > review_above:
        raise ValueError(
            f"Demo {d['demo']} passes but holds still past the review bound."
        )

rows = [f"demo {d['demo']}" for d in demos]
fig = go.Figure()
# The demo's full length, as a track behind its still stretch.
fig.add_bar(
    y=rows,
    x=[d["steps"] for d in demos],
    base=0,
    orientation="h",
    width=0.62,
    marker={"color": RULE, "cornerradius": 4},
    name="Demo length",
    hoverinfo="skip",
)
for decision, color in [("review", REVIEW_COLOR), ("pass", PASS_COLOR)]:
    shown = [d for d in demos if d["decision"] == decision]
    fig.add_bar(
        y=[f"demo {d['demo']}" for d in shown],
        x=[d["end"] - d["start"] for d in shown],
        base=[d["start"] for d in shown],
        orientation="h",
        width=0.38,
        # A paper-coloured edge keeps the stretch apart from its track.
        marker={
            "color": color,
            "cornerradius": 4,
            "line": {"color": PAPER, "width": 2},
        },
        name=f"Holds still ({decision})",
        text=[f"{d['share']:.0f}%" for d in shown],
        textposition="outside",
        textfont={"color": INK},
        cliponaxis=False,
        customdata=[
            [d["channel"], d["start"], d["end"], d["share"], d["decision"]]
            for d in shown
        ],
        hovertemplate="<b>%{y}</b> · %{customdata[0]}<br>still from step %{customdata[1]}"
        " to %{customdata[2]}<br>%{customdata[3]:.1f}% of the demo · %{customdata[4]}"
        "<extra></extra>",
    )
# The bound is a share of each demo, so it lands at a different step on each row.
fig.add_scatter(
    y=rows,
    x=[review_above / 100 * d["steps"] for d in demos],
    mode="markers",
    marker={"symbol": "line-ns", "size": 22, "line": {"color": MUTED, "width": 2}},
    name=f"{review_above:g}% of the demo, Kalanos's review bound",
    hovertemplate=f"<b>%{{y}}</b><br>{review_above:g}% of the demo: step %{{x:.0f}}<extra></extra>",
)

# Point at the passing demo from the right of its track, clear of the other rows.
# Its channel also holds still outside its longest stretch, so the callout
# gives the stretch and the share separately.
for d in demos:
    if d["decision"] == "pass":
        fig.add_annotation(
            x=d["steps"],
            y=f"demo {d['demo']}",
            text=f"<b>Demo {d['demo']}</b>, the only pass<br>{d['channel']} is still"
            f" longest from step {d['start']} to {d['end']},<br>{d['share']:.0f}%"
            f" of the demo in all, under the {review_above:g}% bound",
            ax=70,
            ay=0,
            xanchor="left",
            align="left",
            standoff=6,
            bgcolor=SURFACE,
            bordercolor=RULE,
            borderpad=8,
            arrowcolor=MUTED,
        )

n_review = sum(d["decision"] == "review" for d in demos)
last = max(d["steps"] - d["end"] for d in demos if d["decision"] == "review")
axis = {"gridcolor": RULE, "linecolor": RULE, "zeroline": False, "ticks": ""}
# Room on the right for the callout and the legend.
longest_demo = max(d["steps"] for d in demos)
fig.update_xaxes(
    axis,
    range=[0, longest_demo * 1.8],
    tickvals=list(range(0, longest_demo + 1, 20)),
    title_text=f'<span style="font-family:{MONO}"><b>DATASET</b></span> · action step',
    title_standoff=16,
)
fig.update_yaxes(
    axis,
    categoryorder="array",
    categoryarray=rows,
    autorange="reversed",
    showgrid=False,
)
fig.update_layout(
    barmode="overlay",
    font={
        "family": "'IBM Plex Sans', Arial, sans-serif",
        "size": 15,
        "color": INK,
    },
    paper_bgcolor=PAPER,
    plot_bgcolor=PAPER,
    height=160 + 48 * len(demos),
    margin={"t": 120, "r": 60},
    title={
        "text": f"In {n_review} of {len(demos)} demos, a simulator channel"
        f" holds still until the last {last} steps",
        "font": {"size": 22},
        # Anchored to the plot; the padding holds the two-line subtitle.
        "yref": "paper",
        "y": 1,
        "yanchor": "bottom",
        "pad": {"b": 80},
        "subtitle": {
            "text": "One row per demo: its most-still flagged channel. The channels are"
            " unmapped simulator state,<br>so this is a reason to look, not proof of a fault.",
            "font": {"color": MUTED, "size": 15},
        },
    },
    # The legend sits in the space right of the lower rows, under the callout.
    legend={"title_text": "Kalanos decision", "x": 0.6, "y": 0.4, "yanchor": "top"},
)
fig.show()

## Download the reports

The report as an HTML page and as JSON, zipped.

In [ ]:
# @title Write the HTML report
# The same grade again, written as an HTML page.
# `tail` keeps the card's last lines: the file written, or the error if it failed.
!kalanos --verbosity error grade {SOURCE} --profile numeric-core-v1 --tier standard --report audit.html 2>&1 | tail -n 2

In [ ]:
# @title Download both
import zipfile

from IPython.display import FileLink, display


with zipfile.ZipFile("kalanos-reports.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    archive.write("audit.json")
    archive.write("audit.html")

try:
    # Only Colab has this module; it sends the file to the browser.
    from google.colab import files  # pyright: ignore[reportMissingImports]
except ImportError:
    display(FileLink("kalanos-reports.zip"))
else:
    files.download("kalanos-reports.zip")

**Continue with Kalanos:** [Browse dataset reports](https://hub.kalanos.ai) · [Use the grader locally](https://github.com/KalanosAI/kalanos) · [Attach an audit to a training run](https://github.com/KalanosAI/kalanos-tracking)